## Caricamento dati

Per ora voglio caricare il dataset as-is, cioé senza togliere i duplicati. Non voglio rischiare sottorappresentazione della classe A. Eventualmente voglio affrontare il problema ed eliminare i duplicati sia dal train sia dal test.

In [1]:
import os 
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [5]:
#dataset_dir = '/kaggle/input/FIVES-duplicates'
dataset_dir = '../FIVES'
train_dir = pathlib.Path(dataset_dir) / 'train/Original'
test_dir = pathlib.Path(dataset_dir) / 'test/Original'

Iperparametri

In [6]:
seed = 42
epochs = 500
batch_size = 32
img_size = (2048, 2048)
img_shrinked = (512,512)

In [7]:
def create_dataframe(data_dir):
    filepaths = []
    labels = []
    
    for filename in os.listdir(data_dir):
        if filename.endswith('.png'):
            filepath = os.path.join(data_dir, filename)
            
            # Extract class from filename (format: "121_G.png")
            # Assuming the class is the character after the underscore
            class_name = filename.split('_')[1].split('.')[0]  # Gets 'G' from '121_G.png'
            
            filepaths.append(filepath)
            labels.append(class_name)
    
    return pd.DataFrame({'filepath': filepaths, 'class': labels})

path_n_classes_df = create_dataframe(train_dir)

In [8]:
path_n_classes_df.head()

,filepath,class
0,../FIVES/train/Original/287_D.png,D
1,../FIVES/train/Original/600_N.png,N
2,../FIVES/train/Original/71_A.png,A
3,../FIVES/train/Original/186_D.png,D
4,../FIVES/train/Original/34_A.png,A


In [9]:
# Adds a black box to cover the chinese character on top left corner of an image
def mask_character(img):

    img[:10, :10] = 0

    return img

In [ ]:
train_df, val_df = train_test_split(
    path_n_classes_df, 
    test_size=0.2, 
    stratify=path_n_classes_df['class'],
    random_state=seed
)

# Create data generators
datagen_train = tf.keras.preprocessing.image.ImageDataGenerator(
    #rescale=1./255,
    rotation_range=25,
    horizontal_flip=True,
    preprocessing_function=mask_character,
)


datagen_val = tf.keras.preprocessing.image.ImageDataGenerator(
    #rescale=1./255,
    preprocessing_function=mask_character
)


train_generator = datagen_train.flow_from_dataframe(
    train_df,
    x_col='filepath',
    y_col='class',
    target_size=img_shrinked,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
)

validation_generator = datagen_val.flow_from_dataframe(
    val_df,
    x_col='filepath',
    y_col='class',
    target_size=img_shrinked,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False,
)

In [ ]:
# Get one batch from the generator
images, labels = next(train_generator)

print(f"Images shape: {images.shape}")    # (batch_size, height, width, channels)
print(f"Labels shape: {labels.shape}")    # (batch_size, num_classes)
print(labels)
# Example output:
# Images shape: (32, 224, 224, 3)
# Labels shape: (32, 4)

## Model Loading

In [ ]:
# Monitor F1 score
from sklearn.metrics import f1_score
class F1Callback(tf.keras.callbacks.Callback):
    def __init__(self, x_val, y_val, average="macro"):
        super().__init__()
        self.x_val = x_val
        self.y_val = np.argmax(y_val, axis=1)  
        self.average = average
        self.val_f1 = []

    def on_epoch_end(self, epoch, logs=None):
        y_prob = self.model.predict(self.x_val, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)

        f1 = f1_score(
            self.y_val,
            y_pred,
            average=self.average
        )

        self.val_f1.append(f1)
        print(f" - val_f1: {f1:.4f}")

        if logs is not None:
            logs["val_f1"] = f1

In questa sezione carico la ResNet e la compilo 

In [ ]:
img_shape = img_shrinked + (3, )
img_shape

In [ ]:
resnet = tf.keras.applications.ResNet50(
    input_shape = img_shape,
    include_top = False,
    weights = 'imagenet'
)

resnet.trainable = True

In [ ]:
def create_model(inputs):
    x = tf.keras.applications.resnet50.preprocess_input(inputs)
    x = resnet(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(4, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs)

model = create_model( tf.keras.Input(shape=img_shape) )

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-4),
    loss = tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Extract validation data from generator to use with F1Callback
x_val = []
y_val = []

validation_generator.reset()
for i in range(len(validation_generator)):
    images, labels = validation_generator[i]
    x_val.append(images)
    y_val.append(labels)

x_val = np.concatenate(x_val, axis=0)
y_val = np.concatenate(y_val, axis=0)

print(f"Validation data shape: {x_val.shape}")
print(f"Validation labels shape: {y_val.shape}")

Definizione delle callbacks

In [ ]:
# Calculate f1 on validation set at the end of each epoch
f1_callback = F1Callback(x_val, y_val, average="macro")

# Early stopping on validation f1
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_f1', 
    patience=20,  
    mode='max', 
    restore_best_weights=True,
    start_from_epoch=25
)

# Scheduler
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_f1',
    mode='max',
    patience=8,
    factor=0.5,
    min_lr=1e-7
)

In [ ]:
images, labels = next(train_generator)

print(f"Images shape: {images.shape}")    # (batch_size, height, width, channels)
print(f"Labels shape: {labels.shape}")    # (batch_size, num_classes)
print(f"First label shape: {labels[0].shape}")

In [ ]:
'''# Prendi un batch
images, labels = next(train_generator)

# Configura la visualizzazione
plt.figure(figsize=(15, 10))

# Ottieni i nomi delle classi
class_names = list(train_generator.class_indices.keys())

# Mostra fino a 16 immagini (o quante ne hai nel batch)
num_images = min(16, len(images))

for i in range(num_images):
    plt.subplot(4, 4, i+1)
    plt.imshow(images[i])
    
    # Trova la classe (per one-hot encoded labels)
    class_idx = np.argmax(labels[i])
    class_name = class_names[class_idx]
    
    plt.title(f"Class: {class_name}", fontsize=12)
    plt.axis('off')

plt.tight_layout()
plt.show()'''

In [ ]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=epochs,
    callbacks=[f1_callback, early_stopping, reduce_lr]
)

## Loss and accuracy plotting

In [ ]:
plt.figure(figsize = (10,7))
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='validation accuracy') 
plt.plot(f1_callback.val_f1, label="validation f1")


plt.legend()  
plt.ylim([0,1]) 
#plt.xticks(range(0,n_epochs))
plt.yticks(np.array(range(10))/10)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title(f'Batch size: {batch_size}')
plt.grid('both', linestyle='--')
plt.tight_layout()
plt.savefig(f'accuracy_bs{batch_size}.pdf')

In [ ]:
plt.figure(figsize = (10,7))

plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='validation loss') 

plt.legend()  
#plt.xticks(range(0,n_epochs))
plt.semilogy()
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title(f'Batch size: {batch_size}')
plt.grid('both', linestyle='--')
plt.tight_layout()
plt.savefig(f'loss_bs{batch_size}.pdf')

## Test set

In [ ]:
test_df = create_dataframe(test_dir)
test_df

In [ ]:
datagen_test = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function = mask_character
)

test_generator = datagen_test.flow_from_dataframe(
    test_df,
    x_col= 'filepath',
    y_col = 'class',
    target_size = img_shrinked,
    batch_size = batch_size,
    class_mode = 'categorical',
    shuffle = False
)

In [ ]:
test_pred = model.predict(test_generator)
print(test_pred.shape)
pred_labels = np.argmax(test_pred, axis = 1)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sn

# true labels
true_labels = test_generator.classes
class_labels = ['A', 'D', 'G', 'N']

print(true_labels)
# accuracy
acc = accuracy_score(true_labels, pred_labels)

print(f"Test Accuracy: {acc:.4f}")

cm = confusion_matrix(true_labels, pred_labels, normalize='pred')
df_cm = pd.DataFrame(cm, index = [i for i in class_labels],
                  columns = [i for i in class_labels])
plt.figure(figsize = (10,7))
sn.heatmap(df_cm, annot=True)
plt.title("Confusion Matrix")
plt.tight_layout()


In [ ]:
model.save(f'resnet_{acc:.4f}.keras')